# Meridian MMM Platform - Quick Start Demo

This notebook demonstrates the complete workflow of the Meridian MMM Platform:

1. **Setup** - Import libraries and configure environment
2. **Data Loading** - Load and validate media and sales data
3. **Prior Configuration** - Set up Bayesian priors
4. **Model Training** - Train the Meridian MMM model
5. **Results Analysis** - Analyze ROI, contributions, and model fit
6. **Budget Optimization** - Optimize budget allocation
7. **Scenario Planning** - Compare different budget scenarios

## 1. Setup

In [1]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import sys

# Add src to path
sys.path.insert(0, '../src')

# Import platform modules
from meridian_platform import MediaDataLoader, SalesDataLoader, PriorsConfigLoader
from meridian_platform.modeling.model_runner import MeridianModelRunner
from meridian_platform.optimization.budget_optimizer import BudgetOptimizer

# Configure plotting
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')

print("✅ Setup complete!")

ModuleNotFoundError: No module named 'loguru'

## 2. Generate and Load Sample Data

In [ ]:
# Generate sample data
!python ../scripts/generate_sample_data.py

In [ ]:
# Load media data
media_loader = MediaDataLoader(
    file_path='../data/sample/sample_media_data.csv'
)

media_data = media_loader.load_and_validate()

print(f"\n📺 Media Data Loaded: {len(media_data):,} rows")
print(f"   Touchpoints: {media_data['touchpoint_name'].nunique()}")
print(f"   Geos: {media_data['geo'].nunique()}")
print(f"   Date Range: {media_data['date'].min()} to {media_data['date'].max()}")

media_data.head()

In [ ]:
# Load sales data
sales_loader = SalesDataLoader(
    file_path='../data/sample/sample_sales_data.csv'
)

sales_data = sales_loader.load_and_validate()

print(f"\n💰 Sales Data Loaded: {len(sales_data):,} rows")
print(f"   Total Sales: ${sales_data['sales_value'].sum():,.0f}")
print(f"   Average Weekly Sales: ${sales_data['sales_value'].mean():,.0f}")

sales_data.head()

### Visualize Data

In [ ]:
# Spend by touchpoint
spend_by_touchpoint = media_data.groupby('touchpoint_name')['spend'].sum().sort_values(ascending=False)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

# Spend by channel
spend_by_touchpoint.plot(kind='bar', ax=ax1, color='steelblue')
ax1.set_title('Total Spend by Touchpoint', fontsize=14, fontweight='bold')
ax1.set_ylabel('Spend ($)')
ax1.tick_params(axis='x', rotation=45)

# Sales over time
sales_over_time = sales_data.groupby('date')['sales_value'].sum()
sales_over_time.plot(ax=ax2, color='darkgreen', linewidth=2)
ax2.set_title('Sales Over Time', fontsize=14, fontweight='bold')
ax2.set_ylabel('Sales ($)')

plt.tight_layout()
plt.show()

## 3. Configure Bayesian Priors

In [ ]:
# Load priors configuration
priors_config = PriorsConfigLoader('../config/priors_template.yaml')
priors_config.load_config()

# Validate configuration
validation = priors_config.validate_config()

if validation['valid']:
    print("✅ Priors configuration is valid")
else:
    print("❌ Validation failed:")
    for error in validation['errors']:
        print(f"  - {error}")

In [ ]:
# View ROI priors for our channels
channels = media_data['touchpoint_name'].unique().tolist()
roi_priors = priors_config.get_roi_priors(channels)

print("\n📊 ROI Priors by Channel:\n")
for channel, params in roi_priors.items():
    expected_roi = np.exp(params['loc'])
    print(f"  {channel:20s}: Expected ROI = {expected_roi:.2%} (loc={params['loc']:.2f}, scale={params['scale']:.2f})")

## 4. Train Meridian Model

In [ ]:
# Initialize model runner
runner = MeridianModelRunner(
    media_data=media_data,
    sales_data=sales_data,
    priors_config=priors_config
)

print("✅ Model runner initialized")

In [ ]:
# Prepare data
input_data = runner.prepare_data()

print("✅ Data prepared for Meridian")
print(f"   KPI shape: {input_data.kpi.shape}")
print(f"   Media shape: {input_data.media.shape}")

In [ ]:
# Create model specification
model_spec = runner.create_model_spec()

print("✅ Model specification created")

In [ ]:
# Train model
# NOTE: This can take 5-30 minutes depending on your hardware
# Use GPU if available for much faster training

model = runner.train(
    n_warmup=1000,
    n_samples=1000,
    n_chains=2,
    seed=42
)

print("\n✅ Model training complete!")

## 5. Analyze Results

In [ ]:
# Get convergence diagnostics
diagnostics = runner.get_diagnostics()

print("\n📊 Convergence Diagnostics:\n")
print(f"  R-hat (max): {diagnostics['summary']['r_hat_max']:.4f} (should be < 1.05)")
print(f"  ESS (min):   {diagnostics['summary']['ess_bulk_min']:.0f} (should be > 400)")
print(f"\n  Converged: {'✅ Yes' if diagnostics['convergence']['overall_converged'] else '❌ No'}")

In [ ]:
# Get model results
results = runner.get_results()

print("\n💰 ROI by Channel (Adjusted for Coverage):\n")

if 'roi_adjusted' in results:
    roi_df = pd.DataFrame([
        {'Channel': ch, 'ROI': roi, 'ROI %': f"{roi*100:.1f}%"}
        for ch, roi in results['roi_adjusted'].items()
    ]).sort_values('ROI', ascending=False)
    
    print(roi_df.to_string(index=False))
else:
    print("ROI results not available")

In [ ]:
# Visualize ROI
if 'roi_adjusted' in results:
    fig, ax = plt.subplots(figsize=(12, 6))
    
    roi_df.plot(kind='barh', x='Channel', y='ROI', ax=ax, legend=False, color='steelblue')
    ax.set_title('Return on Investment (ROI) by Channel', fontsize=16, fontweight='bold')
    ax.set_xlabel('ROI (Sales per $ Spent)', fontsize=12)
    ax.axvline(x=1.0, color='red', linestyle='--', linewidth=2, label='Break-even')
    ax.legend()
    
    plt.tight_layout()
    plt.show()

## 6. Budget Optimization

In [ ]:
# Initialize optimizer
optimizer = BudgetOptimizer(trained_model=runner.model)

print("✅ Optimizer initialized")

In [ ]:
# Optimize for maximum ROI
total_budget = 1_000_000  # $1M budget

optimal_result = optimizer.optimize_roi(
    total_budget=total_budget,
    constraints='medium'  # ±50% from current
)

print(f"\n💡 Optimal Budget Allocation (${total_budget:,.0f} budget):\n")

allocation_df = pd.DataFrame([
    {'Channel': ch, 'Budget': f"${budget:,.0f}", 'Budget ($)': budget}
    for ch, budget in optimal_result['allocation'].items()
]).sort_values('Budget ($)', ascending=False)

print(allocation_df[['Channel', 'Budget']].to_string(index=False))

print(f"\n📈 Expected Results:")
print(f"   Total Sales: ${optimal_result['metrics']['total_sales']:,.0f}")
print(f"   Overall ROI: {optimal_result['metrics']['total_roi']:.2%}")

In [ ]:
# Visualize optimal allocation
fig, ax = plt.subplots(figsize=(12, 6))

allocation_df.plot(kind='bar', x='Channel', y='Budget ($)', ax=ax, legend=False, color='green')
ax.set_title('Optimized Budget Allocation', fontsize=16, fontweight='bold')
ax.set_ylabel('Budget ($)', fontsize=12)
ax.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

## 7. Scenario Comparison

In [ ]:
# Create different scenarios
scenarios = [
    {
        'name': 'Current Budget',
        'allocation': {ch: total_budget / len(channels) for ch in channels}
    },
    {
        'name': 'Optimized (Medium)',
        'allocation': optimal_result['allocation']
    },
    {
        'name': 'Digital Focus',
        'allocation': {
            ch: total_budget * 0.3 if 'Digital' in ch or 'Search' in ch or 'Facebook' in ch or 'YouTube' in ch
            else total_budget * 0.1
            for ch in channels
        }
    }
]

# Compare scenarios
comparison = optimizer.compare_scenarios(scenarios)

print("\n📊 Scenario Comparison:\n")
print(comparison[['scenario', 'total_spend', 'total_sales', 'total_roi']].to_string(index=False))

## Summary

This notebook demonstrated:

✅ Loading and validating media and sales data  
✅ Configuring Bayesian priors  
✅ Training a Meridian MMM model  
✅ Analyzing model results and ROI  
✅ Optimizing budget allocation  
✅ Comparing different scenarios  

**Next Steps:**
- Experiment with different prior configurations
- Try different constraint levels in optimization
- Add more channels or geos to your data
- Implement response curve visualization
- Set up automated model refresh pipeline